In [ ]:
Sys.setenv(SP_EVOLUTION_STATUS = "2")
set.seed(1)

library(SeuratObject)
library(Matrix)
library(dplyr)
library(tidyr)
library(readr)
library(Seurat)
library(DESeq2)

In [ ]:
# ---- 1. 参数设置 ------------------------------------------------------------
date <- "2026-07-28_adult"
result_dir <- "/project/work/2026-06-10_adult_DEG/Data/"

In [ ]:
# TODO: 请根据实际数据路径修改
seurat_rds_path <- '/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds'

min_cells <- 30      # 每个 pseudobulk 组的最小细胞数
max_cell <- Inf     # 每个 pseudobulk 组的最大细胞数（降采样上限）
padj_cutoff <- 0.05
logfc_cutoff <- 0.5

# 全局统一的细胞类型顺序（与胚胎保持一致，去掉 RPC）
celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG", "RGC")

# 定义 stage
young_ages <- c("21years", "25years",'30years','32years')
old_ages <- c("67years", "75years", "86years")
unpaired_old_ages <- c("70years")  # 无 peripheral 对照，仅 Central-only 描述

In [ ]:
# ---- 6. DESeq2 区域差异分析 (Central vs Peripheral，按 stage 分别运行) -----
cat("\n=== 5. DESeq2 区域差异分析（Central vs Peripheral，按 stage）===\n")

dir.create(paste0(result_dir, "/02_regional_degs"), recursive = TRUE, showWarnings = FALSE)

In [ ]:
deg_summary <- data.frame(
  celltype = character(),
  stage = character(),
  n_donors_paired = integer(),
  n_genes_tested = integer(),
  Central_up = integer(),
  peripheral_up = integer(),
  stringsAsFactors = FALSE
)

In [ ]:
for (ct in celltype_order) {
  cat("  处理 celltype:", ct, "\n")
  
  pb_file <- paste0(result_dir, "/01_pseudobulk/regional/", "2026-07-28_adult", "_", ct, "_regional_pseudobulk.rds")
  if (!file.exists(pb_file)) {
    cat("    跳过: pseudobulk 文件不存在\n")
    next
  }
  
  pb <- readRDS(pb_file)
  countData_all <- as.matrix(pb$counts)
  colData_all <- pb$colData
  colData_all$group <- rownames(colData_all)
  
  # 只保留 Young 和 Old（排除 Old_unpaired）
  stages_to_test <- c("Young", "Old")
  
  for (ag in stages_to_test) {
    cat("    stage:", ag, "\n")
    
    # 1. 筛选该 stage
    colData_age <- colData_all %>% filter(stage == ag)
    
    if (nrow(colData_age) == 0) {
      cat("      跳过: 该 stage 无数据\n")
      next
    }
    
    # 2. 检查该 stage 内哪些 donor_id 有 Central + peripheral 配对
    donor_tissue <- colData_age %>%
      dplyr::group_by(donor_id, tissue_group) %>%
      dplyr::summarise(n = dplyr::n(), .groups = "drop")
    
    valid_donors <- donor_tissue %>%
      dplyr::count(donor_id) %>%
      dplyr::filter(n == 2) %>%
      dplyr::pull(donor_id)
    
    if (length(valid_donors) < 2) {
      cat("      警告: 有效配对 donor 不足 (", length(valid_donors), "个), 跳过\n")
      next
    }
    
    cat("      有效配对 donor:", paste(valid_donors, collapse = ", "), "\n")
    
    # 3. 只保留配对 donor
    colData <- colData_age %>%
      filter(donor_id %in% valid_donors) %>%
      mutate(
        donor_id = factor(donor_id),
        tissue_group = factor(tissue_group, levels = c("peripheral", "Central"))
      )
    countData <- countData_all[, colData$group, drop = FALSE]
    
    # 4. 过滤低表达基因
    keep <- rowSums(countData) >= 10
    countData <- countData[keep, , drop = FALSE]
    
    if (nrow(countData) == 0 || ncol(countData) == 0) {
      cat("      警告: 过滤后无数据，跳过\n")
      next
    }
    
    # 5. 运行 DESeq2
    # 注意：同一 stage 内 donor_age 相同或相近，用 donor_id 作为配对因子
    dds <- DESeqDataSetFromMatrix(
      countData = countData,
      colData = colData,
      design = ~ donor_id + tissue_group
    )
    
    dds <- DESeq(dds)
    res <- results(dds, contrast = c("tissue_group", "Central", "peripheral"))
    
    res_df <- as.data.frame(res) %>%
      mutate(gene = rownames(.)) %>%
      relocate(gene) %>%
      mutate(
        celltype = ct,
        stage = ag,
        .before = 1
      )
    
    # 6. 保存结果
    write.csv(
      res_df,
      paste0(result_dir, "/02_regional_degs/", date, "_", ct, "_", ag, "_Central_vs_peripheral.csv"),
      row.names = FALSE
    )
    
    # 7. 统计
    n_central <- sum(!is.na(res_df$padj) & res_df$padj < padj_cutoff & !is.na(res_df$log2FoldChange) & res_df$log2FoldChange > logfc_cutoff, na.rm = TRUE)
    n_peripheral <- sum(!is.na(res_df$padj) & res_df$padj < padj_cutoff & !is.na(res_df$log2FoldChange) & res_df$log2FoldChange < -logfc_cutoff, na.rm = TRUE)
    
    deg_summary <- deg_summary %>%
      add_row(
        celltype = ct,
        stage = ag,
        n_donors_paired = length(valid_donors),
        n_genes_tested = nrow(res_df),
        Central_up = n_central,
        peripheral_up = n_peripheral
      )
    
    cat("      完成: 测试 ", nrow(res_df), " 个基因; Central_up:", n_central, "; peripheral_up:", n_peripheral, "\n")
  }
}

In [ ]:
# 保存 DEG 统计摘要
write.csv(
  deg_summary,
  paste0(result_dir, date, "_regional_degs_summary.csv"),
  row.names = FALSE
)
cat("\n  DEG 统计摘要已保存: ", paste0(result_dir, date, "_regional_degs_summary.csv"), "\n")
print(deg_summary)

In [ ]:
# ---- 7. 合并 Regional DEGs --------------------------------------------------
cat("\n========================================\n")
cat("=== 6. 提取严格差异基因 + 合并结果 ===\n")
cat("========================================\n")

reg_dir <- paste0(result_dir, "/02_regional_degs")
reg_files <- list.files(reg_dir, pattern = paste0(date, "_.*_Central_vs_peripheral\\.csv$"), full.names = TRUE)

In [ ]:
if (length(reg_files) == 0) {
  cat("  警告: 未找到 regional DEG 文件\n")
} else {
  reg_list <- lapply(reg_files, function(f) {
    cat("  读取:", basename(f), "\n")
    df <- read.csv(f)
    
    # 从文件名解析 celltype 和 stage
    # 文件名格式: {date}_{celltype}_{stage}_Central_vs_peripheral.csv
    core <- gsub(paste0("^", date, "_"), "", gsub("\\.csv$", "", basename(f)))
    core <- gsub("_Central_vs_peripheral$", "", core)
    
    # 从末尾提取 stage
    stage_candidates <- c("Young", "Old", "Old_unpaired")
    ag <- NA_character_
    for (candidate in stage_candidates) {
      if (endsWith(core, paste0("_", candidate))) {
        ag <- candidate
        ct <- substr(core, 1, nchar(core) - nchar(candidate) - 1)
        break
      }
    }
    
    if (is.na(ag)) {
      stop("无法从文件名解析 stage: ", basename(f))
    }
    
    df_filt <- df %>%
      dplyr::filter(
        !is.na(padj),
        padj < padj_cutoff,
        !is.na(log2FoldChange),
        abs(log2FoldChange) > logfc_cutoff
      ) %>%
      dplyr::mutate(
        celltype = ct,
        stage = ag,
        pair = "Central_vs_peripheral",
        direction = dplyr::if_else(log2FoldChange > 0, "Central_up", "peripheral_up")
      )
    
    return(df_filt)
  })
  
  regional_merged <- bind_rows(reg_list)
  
  # 按项目统一顺序排序 celltype
  regional_merged$celltype <- factor(regional_merged$celltype, levels = celltype_order)
  regional_merged$stage <- factor(regional_merged$stage, levels = c("Young", "Old", "Old_unpaired"))
  regional_merged <- regional_merged %>%
    dplyr::arrange(celltype, stage, direction, padj)
  
  out_reg <- paste0(reg_dir, "/", date, "_regional_degs_filtered_merged.csv")
  write.csv(regional_merged, out_reg, row.names = FALSE)
  cat("\n  完成: 合并", nrow(regional_merged), "条记录 ->", out_reg, "\n")
  
  # 输出方向统计
  cat("\n  方向统计:\n")
  print(regional_merged %>% dplyr::count(celltype, stage, direction))
}

In [ ]:
cat("\n========================================\n")
cat("全部分析完成！\n")
cat("========================================\n")
cat("输出目录结构:\n")
cat("  ", result_dir, "\n")
cat("    |- ", date, "_cell_counts_by_group.csv          # 原始细胞数统计
", sep = "")
cat("    |- ", date, "_pairing_completeness.csv          # 配对完整性检查（含 Old_unpaired 标记）
", sep = "")
cat("    |- ", date, "_low_count_groups.csv              # 低细胞数组合（如有）
", sep = "")
cat("    |- ", date, "_regional_degs_summary.csv         # 各 stage DEG 数量摘要
", sep = "")
cat("    |- 01_pseudobulk/regional/                      # Regional pseudobulk（含 70y）
")
cat("    |- 02_regional_degs/                            # Regional DEG 结果
")
cat("        |- ", date, "_{celltype}_Young_Central_vs_peripheral.csv
", sep = "")
cat("        |- ", date, "_{celltype}_Old_Central_vs_peripheral.csv
", sep = "")
cat("        |- ", date, "_regional_degs_filtered_merged.csv  # 合并后的过滤 DEG
", sep = "")

In [ ]:
regional_merged$celltype_stage <- paste0(regional_merged$celltype,"_",regional_merged$stage)